# Text Ranking

> Everything to know about ordering documents against a query: BM25 and why it refuses to die, dense retrieval, hybrid fusion, cross-encoder reranking, and runnable code that builds the whole two-stage pipeline on a real IR benchmark and measures every stage.

- skip_showdoc: true
- skip_exec: true

## 1. What is Text Ranking?

Text ranking orders a set of documents by relevance to a query. It is the retrieval half of search, RAG, recommendation and question answering - and in a RAG system it is usually the half that decides whether the product works, because a generator cannot answer from a passage that was never fetched.

**The task has a standard two-stage shape**, and understanding why is most of understanding the field:

| Stage | Scores | Cost | Typical model |
|---|---|---|---|
| **Retrieval** (first stage) | query vs **all** N documents | must be sublinear or cheap-per-doc | BM25, bi-encoder + ANN index |
| **Reranking** (second stage) | query vs the top ~50-100 | N is small, so cost per pair can be high | cross-encoder, LLM reranker |

The first stage must be cheap because it touches everything, so it uses representations that can be precomputed and indexed. The second stage can afford to read the query and the document **together**, which is strictly more informative. Retrieval maximises recall; reranking maximises precision at the top. Neither substitutes for the other: rerankers cannot recover a document retrieval missed, and retrieval alone puts mediocre results at rank 1.

**The families of scorer:**

| Family | Representation | Index size | Quality |
|---|---|---|---|
| Lexical (BM25) | Sparse term postings | Small | Strong on rare terms, exact matches, names |
| Dense bi-encoder | 1 vector per document | 4 KB/doc at 1024 dims fp32 | Strong on paraphrase, weak on rare terms |
| Late interaction (ColBERT) | 1 vector per **token** | ~100x a bi-encoder | Between bi- and cross-encoder |
| Cross-encoder | None - computed per pair | n/a | Best; cannot be indexed |
| LLM reranker | None - prompted per pair or list | n/a | Best on nuance; most expensive |

---

## 2. Real-World Use Cases

| Use case | Domain | Consumes / produces | Dominant constraint |
|---|---|---|---|
| RAG context selection | Every LLM application | Question + corpus -> top-k chunks | recall@k caps end-to-end answer quality; latency budget shared with the LLM |
| Web and site search | Search, e-commerce | Query + index -> ranked results | Sub-100 ms at scale; freshness; click feedback loops |
| Product search | E-commerce | Query + catalogue -> ranked SKUs | Exact attribute matching (size, brand); revenue-weighted ranking |
| Enterprise knowledge search | Any large company | Query + documents -> ranked passages | Permissions per document; heterogeneous formats |
| Code search | Developer tools | NL query or snippet -> files | Identifier matching (lexical wins); repo scale |
| Legal and patent discovery | Legal | Query + case law -> ranked documents | **Recall** above all; auditable process |
| Support article routing | Customer service | Ticket -> candidate articles | Deflection rate; short, noisy queries |
| Recommendation reranking | Media, commerce | Candidate set + context -> ordering | Diversity, freshness and business rules on top of relevance |

What the leaderboard number hides:

- **Recall@k in stage one is the hard ceiling on everything downstream.** A reranker that is 10 points better cannot fix a retriever that missed the document. Measure and fix recall first; it is the most common silent failure in RAG.
- **Chunking is part of the ranker.** How you split documents changes what can be retrieved at all. Chunks that lose their heading, or split a table in half, are unretrievable no matter what model scores them.
- **BM25 is not a baseline you can skip.** On exact terms, product codes, names and jargon it beats dense retrieval routinely, which is why every serious system in 2026 is hybrid.
- **Relevance judgements are incomplete.** IR benchmarks label a small pool of documents per query; an unjudged document counts as irrelevant even when it is perfect. This systematically *understates* new systems that surface documents the original pooling never saw.
- **Latency is a stage-by-stage budget.** Reranking 100 candidates with a cross-encoder is 100 forward passes on the critical path. That is the number to check before promising a p95.

---

## 3. How Modern Text Ranking Works

1. **TF-IDF and BM25 (1970s-1994).** Score by term frequency, damped, weighted by inverse document frequency and normalised for document length. BM25 is thirty years old, has two tunable constants, needs no training, and is still competitive on out-of-domain retrieval. Any dense system that cannot beat it is not working.
2. **Learning to rank (2005-2015).** Gradient-boosted trees over hand-made features (BM25 scores, click logs, PageRank, freshness). LambdaMART is still what a lot of production search actually runs for the final ordering, with neural scores as input features.
3. **Neural rerankers (2019).** monoBERT: feed `[CLS] query [SEP] document [SEP]` to BERT and read one relevance logit. A large jump in quality on MS MARCO, and the origin of every cross-encoder reranker used today.
4. **Dense retrieval (2020).** DPR trained separate query and passage encoders contrastively so retrieval became approximate nearest neighbour over vectors. Paraphrase and vocabulary mismatch stopped being fatal; exact rare terms became a weakness.
5. **Late interaction (2020-2022).** ColBERT keeps one vector per token and scores with MaxSim over token pairs - most of a cross-encoder's quality with a precomputable index, at ~100x the storage. See `Multimodal/07_Visual_Document_Retrieval` for the full treatment.
6. **Learned sparse (2021-2023).** SPLADE predicts term weights over the *vocabulary*, including terms not in the document, giving expansion and exact matching in one inverted index. The pragmatic middle between BM25 and dense.
7. **Hybrid as the default (2022-present).** Combine lexical and dense candidate lists with reciprocal rank fusion or a score blend. Consistently better than either alone, and it needs no training - the highest quality-per-effort change available in a retrieval system.
8. **LLM rerankers (2023-2026).** Pointwise ("is this relevant? yes/no", scored from the token logits, as Qwen3-Reranker does), pairwise, or listwise (RankGPT: give the model 20 documents and ask for the order). Best quality, highest cost, and the current top of the BEIR-style boards.

**Where it stands (mid-2026).** The standard architecture is: hybrid BM25 + dense retrieval to get 50-100 candidates, then a cross-encoder or LLM reranker to order the top 10. The open questions are cost (rerankers are expensive per query), long documents, and evaluation on incomplete judgements.

---

## 4. Evaluation Metrics

**nDCG@k** - the headline metric, and the one that handles **graded** relevance (NFCorpus below uses 0/1/2):

$$\text{DCG@}k = \sum_{i=1}^{k} \frac{2^{rel_i} - 1}{\log_2(i + 1)}, \qquad \text{nDCG@}k = \frac{\text{DCG@}k}{\text{IDCG@}k}$$

The `log2(i+1)` discount encodes "rank 1 matters much more than rank 10", and dividing by the ideal ordering (IDCG) normalises to [0, 1] so queries with different numbers of relevant documents can be averaged.

**Recall@k** - what fraction of the relevant documents made it into the top k. **This is the metric for stage one.** If recall@100 is 0.6, then 40% of the answers are unreachable no matter how good the reranker is.

**MRR@k** - the reciprocal rank of the first relevant document, averaged. The right metric when there is exactly one right answer and the user stops at it (navigational search, question answering).

**MAP** - mean average precision over all relevant documents. Binary relevance, rewards getting *all* of them high.

**Pitfalls:**

- **Always report k, and report the stage.** "nDCG 0.35" is meaningless; "nDCG@10 after reranking the BM25 top-100" is a result.
- **Unjudged documents count as irrelevant.** With pooled judgements this penalises systems that find things the pool missed - a systematic bias against new methods.
- **Cutting stage one too tight caps stage two invisibly.** Report recall@k of the retriever separately from nDCG@10 of the pipeline, or you cannot tell which half to fix.
- **Averaging hides variance.** A mean nDCG of 0.35 can be most queries at 0.5 and a fifth at 0. Look at the per-query distribution before optimising.

---

In [ ]:
import math


def dcg(relevances):
    "Discounted cumulative gain with the exponential gain used by TREC and BEIR."
    return sum((2 ** rel - 1) / math.log2(i + 2) for i, rel in enumerate(relevances))


def ndcg_at_k(ranked_ids, relevance, k=10):
    "nDCG@k for one query. `relevance` maps doc_id -> graded relevance (missing = 0)."
    gains = [relevance.get(doc_id, 0) for doc_id in ranked_ids[:k]]
    ideal = sorted(relevance.values(), reverse=True)[:k]
    idcg = dcg(ideal)
    return dcg(gains) / idcg if idcg > 0 else 0.0


def recall_at_k(ranked_ids, relevance, k=100):
    "Fraction of relevant documents retrieved in the top k - the stage-one metric."
    relevant = {d for d, r in relevance.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(ranked_ids[:k])) / len(relevant)


def mrr_at_k(ranked_ids, relevance, k=10):
    "Reciprocal rank of the first relevant document."
    for i, doc_id in enumerate(ranked_ids[:k]):
        if relevance.get(doc_id, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def evaluate(rankings, qrels, k_ndcg=10, k_recall=100):
    "Mean nDCG@k, recall@k and MRR@k over all queries."
    n = len(rankings)
    return {
        f"ndcg@{k_ndcg}": sum(ndcg_at_k(r, qrels[q], k_ndcg) for q, r in rankings.items()) / n,
        f"recall@{k_recall}": sum(recall_at_k(r, qrels[q], k_recall) for q, r in rankings.items()) / n,
        f"mrr@{k_ndcg}": sum(mrr_at_k(r, qrels[q], k_ndcg) for q, r in rankings.items()) / n,
    }


def reciprocal_rank_fusion(rank_lists, k=60):
    "Combine several ranked lists by summing 1/(k + rank). No score calibration needed.\n\n    This is why RRF is the default hybrid method: BM25 scores and cosine similarities live\n    on incompatible scales, and RRF never looks at the scores - only the positions.\n    "
    scores = {}
    for ranking in rank_lists:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return [d for d, _ in sorted(scores.items(), key=lambda kv: -kv[1])]


# One query, worked by hand: relevance 2 at rank 3 and relevance 1 at rank 1.
rel = {"d1": 1, "d3": 2, "d9": 1}
perfect = ["d3", "d1", "d9", "d2", "d4"]
actual = ["d1", "d2", "d3", "d4", "d9"]
print(f"perfect ordering: nDCG@5 {ndcg_at_k(perfect, rel, 5):.3f}  MRR {mrr_at_k(perfect, rel, 5):.3f}")
print(f"actual  ordering: nDCG@5 {ndcg_at_k(actual, rel, 5):.3f}  MRR {mrr_at_k(actual, rel, 5):.3f}")
print(f"recall@3 of the actual ordering: {recall_at_k(actual, rel, 3):.3f}  "
      f"<- one relevant doc is already out of reach at k=3")
print(f"\nRRF of the two lists: {reciprocal_rank_fusion([perfect, actual])[:5]}")

## 5. Datasets

| Dataset | Contents | Size | Scope | License | Typical use |
|---|---|---|---|---|---|
| [BEIR](https://huggingface.co/BeIR) | 18 zero-shot retrieval datasets | 3k-5M docs each | en | mixed | The standard generalisation benchmark |
| [NFCorpus](https://huggingface.co/datasets/mteb/nfcorpus) | Medical/nutrition queries + PubMed abstracts, graded 0-2 | 3.6k docs / 323 test queries | en | CC BY-SA 4.0 | Small enough to run end to end; used below |
| [MS MARCO passage](https://huggingface.co/datasets/microsoft/ms_marco) | Bing queries + passages | 8.8M passages | en | non-commercial | The training set for nearly every reranker |
| [TREC Deep Learning](https://microsoft.github.io/msmarco/TREC-Deep-Learning) | MS MARCO with deep human judgements | 43-54 queries/year | en | non-commercial | The most trustworthy judgements available |
| [SciFact](https://huggingface.co/datasets/mteb/scifact) | Scientific claims + evidence | 5k docs | en | CC BY-NC 2.0 | Small, fast, claim verification |
| [FiQA-2018](https://huggingface.co/datasets/mteb/fiqa) | Financial questions + answers | 57k docs | en | research | Domain-shift stress test |
| [HotpotQA](https://huggingface.co/BeIR/hotpotqa) | Multi-hop questions | 5.2M docs | en | CC BY-SA 4.0 | Multi-document retrieval |
| [MIRACL](https://huggingface.co/datasets/miracl/miracl) | Multilingual retrieval | 18 languages | 18 langs | Apache 2.0 | Non-English ranking |

This notebook runs the full pipeline on **NFCorpus**: 3,633 documents and 323 test queries, small enough to brute-force every stage on this hardware and see the honest numbers, with **graded** relevance so nDCG has something to grade. It is also genuinely hard - biomedical vocabulary, and queries written by laypeople against clinical abstracts - so scores in the 0.3s are normal rather than a bug. Note the judgements are pooled and incomplete, per section 4.

---

## 6. The Model Landscape (mid-2026)

Leaderboards: [MTEB](https://huggingface.co/spaces/mteb/leaderboard) retrieval and reranking tabs, and [BEIR](https://github.com/beir-cellar/beir) for zero-shot generalisation.

**First stage (retrieval):**

| Model | Params | License | Index | Notes |
|---|---|---|---|---|
| BM25 | 0 | - | inverted | no training, strong on rare terms; implemented below |
| [bge-base-en-v1.5](https://huggingface.co/BAAI/bge-base-en-v1.5) | 109M | MIT | 768-dim dense | the practical default; used below |
| [gte-modernbert-base](https://huggingface.co/Alibaba-NLP/gte-modernbert-base) | 149M | Apache 2.0 | 768-dim dense | 8192-token chunks |
| [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) | 596M | Apache 2.0 | 1024-dim dense | top-tier, more expensive |
| [SPLADE-v3](https://huggingface.co/naver/splade-v3) | 110M | non-commercial | learned sparse | inverted index with expansion |
| [ColBERTv2](https://huggingface.co/colbert-ir/colbertv2.0) | 110M | MIT | multi-vector | late interaction; large index |

**Second stage (reranking):**

| Model | Params | License | Type | Notes |
|---|---|---|---|---|
| [ms-marco-MiniLM-L-6-v2](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2) | 22M | Apache 2.0 | cross-encoder | the cheap standard; used below |
| [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3) | 568M | Apache 2.0 | cross-encoder | multilingual, strong; used below |
| [Qwen3-Reranker-0.6B](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B) | 596M | Apache 2.0 | LLM pointwise | yes/no logit scoring; used below |
| [mxbai-rerank-base-v2](https://huggingface.co/mixedbread-ai/mxbai-rerank-base-v2) | 0.5B | Apache 2.0 | LLM reranker | competitive, permissive |
| RankGPT-style listwise | any LLM | - | listwise prompt | orders a whole window at once; expensive |

**How to choose.** Start with hybrid BM25 + `bge-base-en-v1.5` retrieving 100, then `ms-marco-MiniLM-L-6-v2` reranking to 10 - that pipeline runs on a CPU and beats almost any single-stage system. Upgrade the reranker before the retriever if quality at rank 1 is the complaint; upgrade the retriever (or the chunking) if recall@100 is the complaint.

---

## 7. Setup

Package roles:

- `transformers` (>=5.13) + `torch` - the bi-encoder and the three rerankers
- `accelerate` - device placement
- `datasets` - NFCorpus corpus, queries and qrels
- `pandas` + `pyecharts` - benchmark table and charts

BM25 is implemented inline in about 30 lines rather than pulling in `rank_bm25` - seeing the formula is the point of having a baseline, and it is the system everything else has to beat.

**A `transformers` v5 note.** There is no `text-ranking` pipeline (and the `question-answering`, `translation` and `summarization` ones were removed in v5). Cross-encoder rerankers load as `AutoModelForSequenceClassification` with a single output logit, fed `tokenizer(query, document)`; LLM rerankers load as `AutoModelForCausalLM` and are scored from the logits of a yes/no token. Both are shown below.

The corpus here is small enough for exact brute-force search, which is deliberate: it keeps the notebook about *ranking quality* rather than ANN index tuning. Past ~100k vectors you would put FAISS, Qdrant or pgvector under the dense stage - that changes the latency, not the ordering.

---

In [ ]:
# Everything runs through Hugging Face transformers - BM25 is inline, no rank_bm25.
# %pip install -q torch transformers accelerate datasets pandas pyecharts

In [ ]:
import ctypes
import ctypes.util
import gc
import time
from pathlib import Path

import torch
from dotenv import find_dotenv, load_dotenv

# Knowledge/.env sets HF_TOKEN - authenticated HF Hub requests get higher rate limits
load_dotenv(find_dotenv(usecwd=True))

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
if device != "cpu":
    print(torch.cuda.get_device_name(0))
print("device:", device, "| dtype:", dtype)


def vram(tag=""):
    "Report current GPU memory (allocated / reserved). No-op on CPU."
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"VRAM {tag:22s} {alloc:5.2f} GB allocated / {reserved:5.2f} GB reserved")


def free_memory():
    "Collect garbage and hand freed VRAM back to the CUDA allocator.\n\n    Call right after `del`-ing a model you are done with: `del model; free_memory()`.\n    "
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    # glibc keeps freed CPU allocations in its arenas instead of returning them to the
    # OS, so RSS compounds across sections. malloc_trim(0) hands the arenas back.
    try:
        ctypes.CDLL(ctypes.util.find_library("c") or "libc.so.6").malloc_trim(0)
    except Exception:
        pass


# All downloads go to DL_tasks/datasets/ (gitignored)
DATA_DIR = Path("../../datasets")
DATA_DIR.mkdir(exist_ok=True)
HF_CACHE = str(DATA_DIR / "hf_cache")

In [ ]:
from datasets import load_dataset

corpus_ds = load_dataset("mteb/nfcorpus", "corpus", split="corpus", cache_dir=HF_CACHE)
queries_ds = load_dataset("mteb/nfcorpus", "queries", split="queries", cache_dir=HF_CACHE)
qrels_ds = load_dataset("mteb/nfcorpus", split="test", cache_dir=HF_CACHE)

# Documents: title + abstract, which is what every BEIR baseline indexes.
doc_ids = [r["_id"] for r in corpus_ds]
doc_texts = [f"{r['title']} {r['text']}".strip() for r in corpus_ds]
doc_index = {d: i for i, d in enumerate(doc_ids)}

# Graded relevance: qrels[query_id][doc_id] = 0/1/2
qrels = {}
for r in qrels_ds:
    if r["score"] > 0:
        qrels.setdefault(r["query-id"], {})[r["corpus-id"]] = int(r["score"])

query_text = {r["_id"]: r["text"] for r in queries_ds}

N_QUERIES = 100   # test queries to evaluate (323 have judgements)
query_ids = [q for q in sorted(qrels) if q in query_text][:N_QUERIES]

print(f"corpus : {len(doc_ids):,} documents")
print(f"queries: {len(query_ids)} of {len(qrels)} judged test queries")
print(f"judgements per query: "
      f"{sum(len(qrels[q]) for q in query_ids) / len(query_ids):.1f} relevant documents on average")
qid = query_ids[0]
print(f"\nexample query: {query_text[qid]!r}")
top_rel = sorted(qrels[qid].items(), key=lambda kv: -kv[1])[:2]
for d, rel in top_rel:
    print(f"  relevance {rel}: {doc_texts[doc_index[d]][:120]}...")

## 8. The baseline that must be beaten: BM25

Thirty years old, no training, no GPU, and still the system every retrieval paper has to beat. BM25 scores a document by the query terms it contains, weighted by how rare each term is (IDF), with two corrections that are the whole reason it works better than TF-IDF:

$$\text{score}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

- **Term-frequency saturation** (`k1`, typically 1.2-1.5): the tenth occurrence of a word adds much less than the second. TF-IDF's linear growth over-rewards keyword stuffing.
- **Length normalisation** (`b`, typically 0.75): long documents contain more of everything, so their term counts get discounted.

Its strength is exact matching - names, codes, biomedical terms, anything rare. Its weakness is vocabulary mismatch: a query saying "heart attack" scores zero against a document saying "myocardial infarction". That single sentence is the entire motivation for dense retrieval, and the reason hybrid beats both.

---

In [ ]:
import re
from collections import Counter


class BM25:
    "Okapi BM25. Thirty lines, no dependency, and the baseline everything must beat."

    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [self._tokenise(d) for d in documents]
        self.lengths = [len(d) for d in self.docs]
        self.avgdl = sum(self.lengths) / len(self.docs)
        self.freqs = [Counter(d) for d in self.docs]
        df = Counter(t for doc in self.docs for t in set(doc))
        n = len(self.docs)
        # Robertson-Sparck-Jones IDF with the +0.5 smoothing that keeps it positive.
        self.idf = {t: math.log(1 + (n - c + 0.5) / (c + 0.5)) for t, c in df.items()}

    @staticmethod
    def _tokenise(text):
        return re.findall(r"[a-z0-9]+", text.lower())

    def scores(self, query):
        "BM25 score of every document for one query."
        q_terms = self._tokenise(query)
        out = [0.0] * len(self.docs)
        for t in q_terms:
            idf = self.idf.get(t)
            if idf is None:
                continue
            for i, freq in enumerate(self.freqs):
                f = freq.get(t, 0)
                if f:
                    denom = f + self.k1 * (1 - self.b + self.b * self.lengths[i] / self.avgdl)
                    out[i] += idf * f * (self.k1 + 1) / denom
        return out

    def search(self, query, k=100):
        "Top-k document indices for a query."
        scored = self.scores(query)
        return sorted(range(len(scored)), key=lambda i: -scored[i])[:k]


t0 = time.perf_counter()
bm25 = BM25(doc_texts)
print(f"indexed {len(doc_texts):,} documents in {time.perf_counter() - t0:.1f}s "
      f"(vocabulary {len(bm25.idf):,} terms)")

t0 = time.perf_counter()
bm25_rankings = {q: [doc_ids[i] for i in bm25.search(query_text[q], k=100)] for q in query_ids}
bm25_seconds = time.perf_counter() - t0

bm25_scores = evaluate(bm25_rankings, qrels)
print(f"\nBM25: " + "  ".join(f"{k} {v:.4f}" for k, v in bm25_scores.items())
      + f"   {bm25_seconds:.1f}s for {len(query_ids)} queries")

# The vocabulary-mismatch failure, made concrete.
print("\nBM25 on a paraphrased query:")
for q in ["heart attack prevention diet", "myocardial infarction prevention diet"]:
    top = bm25.search(q, k=1)[0]
    print(f"  {q!r}\n     -> {doc_texts[top][:100]}")

## 9. Dense retrieval

Encode every document once into a vector, encode the query, take the top cosine similarities. Vocabulary mismatch stops mattering because "heart attack" and "myocardial infarction" land near each other in the space.

Two mechanics carried over from `07_Feature_Extraction`, and both are easy to get wrong:

- **The query prefix.** bge models are trained with an instruction on the **query side only**. Encoding documents with the same prefix, or omitting it on queries, costs real recall and produces no error.
- **CLS pooling and L2 normalisation**, because that is what this model was trained with.

The corpus is small enough for an exact matrix multiply, so what you see is the true ranking quality with no ANN approximation in the way. On NFCorpus specifically, expect dense retrieval to be **competitive with rather than dominant over** BM25 - biomedical terminology is exactly the case where exact matching is strong, which is the most useful thing this dataset teaches.

---

In [ ]:
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

emb_tok = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5", cache_dir=HF_CACHE)
emb_model = AutoModel.from_pretrained(
    "BAAI/bge-base-en-v1.5", dtype=dtype, cache_dir=HF_CACHE
).to(device).eval()
vram("bge loaded")


@torch.inference_mode()
def embed(texts, batch_size=64, max_length=256):
    "CLS-pooled, L2-normalised embeddings - the pooling bge-* was trained with."
    vecs = []
    for i in range(0, len(texts), batch_size):
        enc = emb_tok(texts[i:i + batch_size], padding=True, truncation=True,
                      max_length=max_length, return_tensors="pt").to(device)
        hidden = emb_model(**enc).last_hidden_state[:, 0]
        vecs.append(F.normalize(hidden.float(), dim=-1).cpu())
    return torch.cat(vecs)


t0 = time.perf_counter()
doc_vecs = embed(doc_texts)
index_seconds = time.perf_counter() - t0
print(f"indexed {len(doc_texts):,} documents in {index_seconds:.1f}s -> {tuple(doc_vecs.shape)} "
      f"({doc_vecs.numel() * 4 / 1e6:.1f} MB in fp32)")

t0 = time.perf_counter()
q_vecs = embed([QUERY_PREFIX + query_text[q] for q in query_ids])
sims = q_vecs @ doc_vecs.T
dense_rankings = {
    q: [doc_ids[i] for i in sims[row].topk(100).indices.tolist()]
    for row, q in enumerate(query_ids)
}
dense_seconds = time.perf_counter() - t0

dense_scores = evaluate(dense_rankings, qrels)
print(f"\ndense: " + "  ".join(f"{k} {v:.4f}" for k, v in dense_scores.items())
      + f"   {dense_seconds:.1f}s for {len(query_ids)} queries")

# The paraphrase that BM25 could not handle.
pv = embed([QUERY_PREFIX + "heart attack prevention diet",
            QUERY_PREFIX + "myocardial infarction prevention diet"])
overlap = len(set((pv[0] @ doc_vecs.T).topk(10).indices.tolist())
              & set((pv[1] @ doc_vecs.T).topk(10).indices.tolist()))
print(f"\ntop-10 overlap between the two phrasings: {overlap}/10 "
      f"(BM25 shares no query terms between them at all)")

del emb_model
free_memory()
vram("after dense retrieval")

## 10. Hybrid: reciprocal rank fusion

Lexical and dense retrieval fail differently - BM25 misses paraphrase, dense misses rare exact terms - so combining them recovers documents neither finds alone. That is why hybrid is the default in 2026, and it needs no training at all.

The problem is that a BM25 score of 14.2 and a cosine of 0.71 are not comparable, and normalising them into a common range is fragile (min-max normalisation depends on the candidate set, so the same document gets a different fused score depending on what else was retrieved).

**Reciprocal rank fusion** sidesteps this by ignoring scores entirely and using only positions:

$$\text{RRF}(d) = \sum_{\text{systems}} \frac{1}{k + \text{rank}(d)}, \qquad k \approx 60$$

The constant `k` damps the influence of the top ranks so a single system cannot dominate. It is one of the highest quality-per-line-of-code changes available in a retrieval system, and it is the twelve lines in section 4.

Watch **recall@100** in particular here: fusion's real contribution is widening the candidate pool that the reranker in section 11 gets to work with.

---

In [ ]:
hybrid_rankings = {
    q: reciprocal_rank_fusion([bm25_rankings[q], dense_rankings[q]], k=60)[:100]
    for q in query_ids
}
hybrid_scores = evaluate(hybrid_rankings, qrels)
print("BM25   " + "  ".join(f"{k} {v:.4f}" for k, v in bm25_scores.items()))
print("dense  " + "  ".join(f"{k} {v:.4f}" for k, v in dense_scores.items()))
print("hybrid " + "  ".join(f"{k} {v:.4f}" for k, v in hybrid_scores.items()))

# How much each system contributes uniquely - the argument for fusion in one number.
only_bm25 = only_dense = both = 0
for q in query_ids:
    relevant = {d for d, r in qrels[q].items() if r > 0}
    b = relevant & set(bm25_rankings[q][:100])
    d = relevant & set(dense_rankings[q][:100])
    only_bm25 += len(b - d)
    only_dense += len(d - b)
    both += len(b & d)
print(f"\nrelevant documents in the top 100: {both} found by both, "
      f"{only_bm25} only by BM25, {only_dense} only by dense")
print("the two 'only' columns are what fusion recovers, and why single-system retrieval leaves value behind")

## 11. Reranking the shortlist

Now the second stage. Take the top 50 hybrid candidates per query and rescore them with models that read the query and the document **together** - one forward pass per pair, which is only affordable because the candidate set is small.

Three rerankers, spanning the current design space:

- **`ms-marco-MiniLM-L-6-v2`** (22M) - the classic cheap cross-encoder. Six layers, one relevance logit, and it runs on a CPU.
- **`bge-reranker-v2-m3`** (568M) - a large multilingual cross-encoder, same interface, ~25x the compute.
- **`Qwen3-Reranker-0.6B`** - an LLM reranker. It is prompted with the query and the document and asked to answer `yes` or `no`; the score is the log-probability difference between those two tokens. No generation, one forward pass, and the relevance judgement is read straight out of the next-token distribution - the same trick as `00_Text_Classification` section 11, applied to ranking.

The measurement that matters is **nDCG@10 before and after**, with the reranking latency next to it. Reranking is on the critical path of every query, so 50 pairs at 20 ms each is a second of user-visible latency, and that is the real constraint on how deep you rerank.

---

In [ ]:
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification

RERANK_DEPTH = 50   # candidates per query taken from the hybrid list

candidates = {q: hybrid_rankings[q][:RERANK_DEPTH] for q in query_ids}
doc_text_by_id = dict(zip(doc_ids, doc_texts))


@torch.inference_mode()
def rerank_cross_encoder(model, tok, batch_size=32, max_length=320):
    "Rescore each query's candidates with a cross-encoder; return new rankings and seconds."
    rankings, t0 = {}, time.perf_counter()
    for q in query_ids:
        docs = candidates[q]
        scores = []
        for i in range(0, len(docs), batch_size):
            chunk = docs[i:i + batch_size]
            enc = tok([query_text[q]] * len(chunk), [doc_text_by_id[d] for d in chunk],
                      padding=True, truncation=True, max_length=max_length,
                      return_tensors="pt").to(device)
            scores.extend(model(**enc).logits.float().squeeze(-1).tolist())
        rankings[q] = [d for _, d in sorted(zip(scores, docs), key=lambda kv: -kv[0])]
    return rankings, time.perf_counter() - t0


rerank_results = {}

for name, model_id in [("ms-marco-MiniLM-L6", "cross-encoder/ms-marco-MiniLM-L-6-v2"),
                       ("bge-reranker-v2-m3", "BAAI/bge-reranker-v2-m3")]:
    ce_tok = AutoTokenizer.from_pretrained(model_id, cache_dir=HF_CACHE)
    ce = AutoModelForSequenceClassification.from_pretrained(
        model_id, dtype=dtype, cache_dir=HF_CACHE
    ).to(device).eval()
    ranked, seconds = rerank_cross_encoder(ce, ce_tok)
    scores = evaluate(ranked, qrels)
    rerank_results[name] = {**scores, "seconds": seconds}
    print(f"{name:20s} " + "  ".join(f"{k} {v:.4f}" for k, v in scores.items())
          + f"   {seconds:.1f}s ({len(query_ids) * RERANK_DEPTH} pairs)")
    del ce, ce_tok      # free each model before loading the next so VRAM stays flat
    free_memory()

vram("after cross-encoders")

In [ ]:
# The LLM reranker: score the 'yes' token against the 'no' token, one forward pass per pair.
qr_id = "Qwen/Qwen3-Reranker-0.6B"
qr_tok = AutoTokenizer.from_pretrained(qr_id, padding_side="left", cache_dir=HF_CACHE)
qr = AutoModelForCausalLM.from_pretrained(
    qr_id, dtype=dtype, device_map=device, cache_dir=HF_CACHE
).eval()
vram("qwen3-reranker loaded")

YES_ID = qr_tok.convert_tokens_to_ids("yes")
NO_ID = qr_tok.convert_tokens_to_ids("no")
INSTRUCT = ("Judge whether the Document meets the requirements based on the Query. "
            "Answer only \"yes\" or \"no\".")


def rerank_prompt(query, document, max_chars=1200):
    "The pointwise relevance prompt; the answer is read from the logits, never generated."
    return qr_tok.apply_chat_template(
        [{"role": "system", "content": INSTRUCT},
         {"role": "user", "content": f"<Query>: {query}\n<Document>: {document[:max_chars]}"}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )


@torch.inference_mode()
def rerank_llm(batch_size=8):
    "Rescore candidates by log P(yes) - log P(no) at the first generated position."
    rankings, t0 = {}, time.perf_counter()
    for q in query_ids:
        docs = candidates[q]
        scores = []
        for i in range(0, len(docs), batch_size):
            prompts = [rerank_prompt(query_text[q], doc_text_by_id[d]) for d in docs[i:i + batch_size]]
            enc = qr_tok(prompts, return_tensors="pt", padding=True, truncation=True,
                         max_length=512).to(qr.device)
            logits = qr(**enc).logits[:, -1, :].float()
            log_probs = torch.log_softmax(logits, dim=-1)
            scores.extend((log_probs[:, YES_ID] - log_probs[:, NO_ID]).tolist())
        rankings[q] = [d for _, d in sorted(zip(scores, docs), key=lambda kv: -kv[0])]
    return rankings, time.perf_counter() - t0


llm_ranked, llm_seconds = rerank_llm()
llm_scores = evaluate(llm_ranked, qrels)
rerank_results["qwen3-reranker-0.6b"] = {**llm_scores, "seconds": llm_seconds}
print("qwen3-reranker-0.6b  " + "  ".join(f"{k} {v:.4f}" for k, v in llm_scores.items())
      + f"   {llm_seconds:.1f}s")

del qr, qr_tok
free_memory()
vram("after llm reranker")

## 12. Head-to-head Benchmark

Every stage of the pipeline on the same 100 NFCorpus queries and the same corpus.

How to read the table, which is the actual lesson of the notebook:

- **recall@100 changes only in the retrieval rows.** Reranking reorders the candidate list; it cannot add to it. The reranked rows inherit the hybrid retriever's recall exactly, which is the ceiling section 2 warned about, visible as a constant column.
- **nDCG@10 is where reranking pays.** The jump from hybrid retrieval to a reranked hybrid is usually larger than any difference between the retrievers, and it comes from a 22M model.
- **Latency is the price.** The seconds column for a reranker is 5,000 forward passes (100 queries x 50 candidates), on the critical path. Doubling `RERANK_DEPTH` doubles it for a small quality gain - which is how you choose the depth.

NFCorpus is hard and the judgements are incomplete, so absolute values in the 0.3s are normal. One hundred queries also means real variance - treat gaps under ~0.02 nDCG as noise.

---

In [ ]:
import pandas as pd

rows = [
    {"stage": "retrieval", "system": "BM25 (no training)", **bm25_scores, "seconds": bm25_seconds},
    {"stage": "retrieval", "system": "dense (bge-base)", **dense_scores, "seconds": dense_seconds},
    {"stage": "retrieval", "system": "hybrid (RRF)", **hybrid_scores,
     "seconds": bm25_seconds + dense_seconds},
]
for name, res in rerank_results.items():
    rows.append({"stage": f"rerank top-{RERANK_DEPTH}", "system": name, **res})

df = pd.DataFrame(rows)
df = df[["stage", "system", "ndcg@10", "recall@100", "mrr@10", "seconds"]].round(4)
print(df.to_string(index=False))
print(f"\nrecall@100 is identical for every reranked row: a reranker reorders {RERANK_DEPTH} "
      "candidates, it cannot retrieve new ones")

In [ ]:
from pyecharts import options as opts
from pyecharts.charts import Bar

bar = (
    Bar()
    .add_xaxis([r["system"] for r in rows])
    .add_yaxis("nDCG@10 x100", [round(r["ndcg@10"] * 100, 1) for r in rows])
    .add_yaxis("recall@100 x100", [round(r["recall@100"] * 100, 1) for r in rows])
    .add_yaxis("MRR@10 x100", [round(r["mrr@10"] * 100, 1) for r in rows])
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title=f"NFCorpus, {len(query_ids)} test queries",
            subtitle="RTX 3060 - the last three rows rerank the hybrid top-50",
        ),
        yaxis_opts=opts.AxisOpts(name="score x100"),
        xaxis_opts=opts.AxisOpts(name="system", axislabel_opts=opts.LabelOpts(rotate=20)),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        legend_opts=opts.LegendOpts(pos_top="8%"),
    )
)
bar.render_notebook()

In [ ]:
from pyecharts.charts import Scatter

# Quality against latency - the chart that sets the reranking depth.
scatter = Scatter()
scatter.add_xaxis([round(r["seconds"], 2) for r in rows])
for r in rows:
    scatter.add_yaxis(
        r["system"], [[round(r["seconds"], 2), round(r["ndcg@10"] * 100, 1)]],
        symbol_size=18, label_opts=opts.LabelOpts(is_show=False),
    )
scatter.set_global_opts(
    title_opts=opts.TitleOpts(title=f"nDCG@10 vs seconds for {len(query_ids)} queries"),
    xaxis_opts=opts.AxisOpts(name="seconds (log-ish scale in practice)", type_="value"),
    yaxis_opts=opts.AxisOpts(name="nDCG@10 x100", type_="value"),
    tooltip_opts=opts.TooltipOpts(trigger="item"),
)
scatter.render_notebook()

## 13. Interactive: rank your own documents

The whole pipeline over your own corpus: BM25, dense, fused, then reranked, with each stage's ordering printed so the differences are visible on text you understand. This is the cell people run on its own, so it opens with a `require(...)` guard naming what it needs from Setup rather than dying on a bare `NameError`.

The queries to try, because each isolates one stage:

- **A rare exact term** (an error code, a product name). BM25 should win and dense should struggle.
- **A pure paraphrase** with no shared vocabulary. The reverse.
- **A query where the best document is only *partly* about the topic.** This is what the reranker is for, and where retrieval alone puts the wrong thing first.

---

In [ ]:
def require(*names):
    "Fail early and clearly if the notebook's setup / helper cells have not been run."
    missing = [n for n in names if n not in globals()]
    if missing:
        raise NameError(
            f"this demo needs {', '.join(missing)} from earlier in the notebook. "
            "Run the setup and helper cells first (Run > Run All Above Selected Cell)."
        )


require("device", "dtype", "HF_CACHE", "free_memory", "vram", "BM25",
        "reciprocal_rank_fusion")

import torch.nn.functional as F
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

MY_DOCS = [
    "Error 5041 is raised when the connection pool is exhausted. Increase pool_size or "
    "reduce the number of concurrent workers.",
    "The connection pool is configured in settings.py and defaults to ten connections.",
    "A heart attack, known clinically as a myocardial infarction, occurs when blood flow "
    "to the heart muscle is blocked.",
    "Dietary changes, exercise and statins reduce the risk of cardiovascular events.",
    "The nightly migration finished early because throughput peaked at 41,000 rows per second.",
    "Read replicas lagged during the morning peak and about 3% of requests saw stale data.",
]
MY_QUERIES = [
    "error 5041",
    "what causes a cardiac infarction",
    "why did some users see old data",
]
TOP_K = 3
MY_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

# Re-runnable: this cell frees its models at the end, so guard the loads or a second
# shift-enter raises NameError.
if "my_emb" not in globals():
    my_emb_tok = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5", cache_dir=HF_CACHE)
    my_emb = AutoModel.from_pretrained(
        "BAAI/bge-base-en-v1.5", dtype=dtype, cache_dir=HF_CACHE
    ).to(device).eval()
    my_ce_tok = AutoTokenizer.from_pretrained("cross-encoder/ms-marco-MiniLM-L-6-v2",
                                              cache_dir=HF_CACHE)
    my_ce = AutoModelForSequenceClassification.from_pretrained(
        "cross-encoder/ms-marco-MiniLM-L-6-v2", dtype=dtype, cache_dir=HF_CACHE
    ).to(device).eval()
    vram("live models")


@torch.inference_mode()
def my_embed(texts):
    enc = my_emb_tok(texts, padding=True, truncation=True, max_length=256,
                     return_tensors="pt").to(device)
    return F.normalize(my_emb(**enc).last_hidden_state[:, 0].float(), dim=-1).cpu()


my_bm25 = BM25(MY_DOCS)
my_doc_vecs = my_embed(MY_DOCS)

for query in MY_QUERIES:
    lexical = my_bm25.search(query, k=len(MY_DOCS))
    sims = (my_embed([MY_QUERY_PREFIX + query]) @ my_doc_vecs.T)[0]
    dense = sims.argsort(descending=True).tolist()
    fused = reciprocal_rank_fusion([lexical, dense])

    with torch.inference_mode():
        enc = my_ce_tok([query] * len(fused), [MY_DOCS[i] for i in fused],
                        padding=True, truncation=True, max_length=320,
                        return_tensors="pt").to(device)
        ce_scores = my_ce(**enc).logits.float().squeeze(-1).tolist()
    reranked = [i for _, i in sorted(zip(ce_scores, fused), key=lambda kv: -kv[0])]

    print(f"\nQ: {query}")
    for label, order in [("bm25", lexical), ("dense", dense), ("hybrid", fused),
                         ("reranked", reranked)]:
        print(f"  {label:9s} " + " | ".join(MY_DOCS[i][:42] for i in order[:TOP_K]))

del my_emb, my_emb_tok, my_ce, my_ce_tok
free_memory()
vram("final")

## 14. Going Further

- **Measure recall@k of stage one before touching anything else.** It is the ceiling on the whole system, and in a disappointing RAG deployment it is the problem far more often than the generator is.
- **Fix chunking before fixing models.** Chunk at semantic boundaries, overlap by 10-20%, and prepend the document title and section heading to every chunk. Unretrievable text cannot be reranked.
- **Ship hybrid by default.** BM25 plus dense plus RRF is a few dozen lines and needs no training. Section 10's "only found by one system" counts are the argument.
- **Rerank the top 50, not the top 1000.** The quality curve flattens quickly and the latency does not. Tune the depth against your p95 budget with the chart in section 12.
- **Fine-tune the retriever on your own (query, relevant doc) pairs** with in-batch negatives plus **hard negatives** mined from the current system's top results. This is the highest-value training you can do in retrieval, and a few thousand pairs is enough.
- **Use a real index past ~100k documents.** FAISS, Qdrant, LanceDB or pgvector for the dense side; Elasticsearch/OpenSearch or a SPLADE inverted index for the sparse side. Exact search is fine below that and removes a variable while you debug ranking quality.
- **Mine your logs.** Clicks, dwell time and "was this helpful" answers are a training set nobody else has. Position bias needs correcting, but even a naive click model beats a generic checkpoint.
- **Consider listwise LLM reranking** (RankGPT-style) when quality dominates cost: give the model 20 documents and ask for the ordering. It sees the candidates in context of each other, which pointwise scoring never does.
- **Related notebooks.** `07_Feature_Extraction` (the embeddings this runs on, and the storage arithmetic), `10_Sentence_Similarity` (bi- vs cross-encoders, thresholds), `03_Question_Answering` (reading what you retrieved), `08_Text_Generation` (the generator at the end of RAG), `Multimodal/07_Visual_Document_Retrieval` (late interaction and retrieval over page images).

---